# 03 - Model Comparison

This notebook trains and evaluates all candidate models for NRL match winner prediction
using walk-forward backtesting (strict temporal splits). We compare baseline models
against classical ML models and select the best performer.

**Sections:**
1. Load features and prepare data
2. Setup walk-forward backtesting
3. Run baseline models (home always, ladder, odds-implied, Elo)
4. Train and evaluate classical models (LogReg, RF, XGBoost, LightGBM)
5. Results comparison table
6. Calibration plots for top models
7. Feature importance (SHAP) for best tree model
8. Summary and best model selection

In [ ]:
# ============================================================
# Cell 1: Imports and load features
# ============================================================
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import time

from config.settings import (
    FEATURES_DIR, PROCESSED_DIR, OUTPUTS_DIR, RANDOM_SEED,
    START_YEAR, END_YEAR,
)

# Project modules
from modelling.baseline_models import (
    HomeAlwaysModel, LadderModel, OddsImpliedModel, EloModel,
)
from modelling.classical_models import get_model
from evaluation.backtesting import WalkForwardBacktester
from evaluation.metrics import compute_all_metrics, compare_models

sns.set_theme(style="whitegrid", palette="deep", font_scale=1.1)
plt.rcParams["figure.figsize"] = (14, 7)
plt.rcParams["figure.dpi"] = 120
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# ------------------------------------------------------------------
# Load features
# ------------------------------------------------------------------
features_path = FEATURES_DIR / "features.parquet"
alt_paths = list(FEATURES_DIR.glob("*.parquet")) + list(FEATURES_DIR.glob("*.csv"))

if features_path.exists():
    features = pd.read_parquet(features_path)
elif alt_paths:
    fpath = alt_paths[0]
    features = pd.read_parquet(fpath) if fpath.suffix == ".parquet" else pd.read_csv(fpath)
else:
    raise FileNotFoundError(f"No feature files found in {FEATURES_DIR}. Run the feature pipeline first.")

# Build target if missing
target_col = None
for candidate in ["home_win", "target", "result", "y"]:
    if candidate in features.columns:
        target_col = candidate
        break
if target_col is None and "home_score" in features.columns:
    features["home_win"] = (features["home_score"] > features["away_score"]).astype(int)
    target_col = "home_win"

# Detect season/year column
year_col = "season" if "season" in features.columns else "year"

print(f"Features shape: {features.shape}")
print(f"Target column: {target_col}")
print(f"Year column: {year_col}")
print(f"Seasons: {sorted(features[year_col].dropna().unique())}")

In [ ]:
# ============================================================
# Cell 2: Setup walk-forward backtesting
# ============================================================

# Filter to rows with valid target
valid_mask = features[target_col].notna()
df = features.loc[valid_mask].copy().reset_index(drop=True)
target = df[target_col].astype(int)

# Determine test years (last ~8 seasons for testing)
all_years = sorted(df[year_col].dropna().unique())
# Use the last 8 years as test folds (training on everything before)
test_start = max(int(all_years[0]) + 5, 2018)  # Need at least 5 years for training
test_years = [y for y in all_years if y >= test_start and y <= END_YEAR]

print(f"Training start year: {int(all_years[0])}")
print(f"Test years: {test_years}")
print(f"Total samples: {len(df):,}")

# Setup the backtester
backtester = WalkForwardBacktester(
    train_start_year=int(all_years[0]),
    test_years=test_years,
    expanding=True,
    year_column=year_col,
)

# Identify feature columns (numeric, excluding metadata)
META_COLS = {
    "home_team", "away_team", "venue", "date", "kickoff", "match_id",
    "season", "year", "round", "home_score", "away_score", target_col,
    "time_slot", "is_finals",
}
feature_cols = [
    c for c in df.select_dtypes(include=[np.number]).columns
    if c not in META_COLS and c != year_col
]

print(f"Feature columns for modelling: {len(feature_cols)}")
print(f"Walk-forward folds: {len(test_years)}")

# Print fold details
print("\nFold details:")
for train_idx, test_idx, test_year in backtester.get_splits(df):
    print(f"  Test {test_year}: train={len(train_idx):,} rows, test={len(test_idx):,} rows")

In [ ]:
# ============================================================
# Cell 3: Run baselines and display results
# ============================================================

# We need to handle baselines specially since they require specific columns
# (home_team, away_team, home_odds, away_odds, home_ladder_pos, etc.)
# that may not be in the pure numeric feature matrix.

baseline_results = {}
baseline_predictions = {}

print("Running baseline models...")
print("=" * 60)

# --- 1. Home Always Model ---
print("\n1. HomeAlwaysModel")
t0 = time.time()
results_home, preds_home = backtester.run(
    model_factory=HomeAlwaysModel,
    features_df=df[feature_cols + [year_col]],
    target=target,
    needs_retraining=False,
)
elapsed = time.time() - t0
agg_home = compute_all_metrics(preds_home["y_true"], preds_home["y_pred"], preds_home["y_prob"])
baseline_results["Home Always"] = agg_home
baseline_predictions["Home Always"] = preds_home
print(f"   Accuracy: {agg_home['accuracy']:.4f} | Log-Loss: {agg_home['log_loss']:.4f} | "
      f"Brier: {agg_home['brier_score']:.4f} | AUC: {agg_home['auc_roc']:.4f} | Time: {elapsed:.1f}s")

# --- 2. Ladder Model ---
if "home_ladder_pos" in df.columns and "away_ladder_pos" in df.columns:
    print("\n2. LadderModel")
    t0 = time.time()
    # LadderModel needs the ladder columns in the feature DataFrame
    ladder_feature_cols = feature_cols + [year_col]
    for extra in ["home_ladder_pos", "away_ladder_pos"]:
        if extra not in ladder_feature_cols:
            ladder_feature_cols.append(extra)

    results_ladder, preds_ladder = backtester.run(
        model_factory=LadderModel,
        features_df=df[ladder_feature_cols],
        target=target,
        needs_retraining=False,
    )
    elapsed = time.time() - t0
    agg_ladder = compute_all_metrics(preds_ladder["y_true"], preds_ladder["y_pred"], preds_ladder["y_prob"])
    baseline_results["Ladder"] = agg_ladder
    baseline_predictions["Ladder"] = preds_ladder
    print(f"   Accuracy: {agg_ladder['accuracy']:.4f} | Log-Loss: {agg_ladder['log_loss']:.4f} | "
          f"Brier: {agg_ladder['brier_score']:.4f} | AUC: {agg_ladder['auc_roc']:.4f} | Time: {elapsed:.1f}s")
else:
    print("\n2. LadderModel -- SKIPPED (ladder columns not found)")

# --- 3. Odds Implied Model ---
if "home_odds" in df.columns and "away_odds" in df.columns:
    print("\n3. OddsImpliedModel")
    t0 = time.time()
    odds_feature_cols = feature_cols + [year_col]
    for extra in ["home_odds", "away_odds"]:
        if extra not in odds_feature_cols:
            odds_feature_cols.append(extra)

    results_odds, preds_odds = backtester.run(
        model_factory=OddsImpliedModel,
        features_df=df[odds_feature_cols],
        target=target,
        needs_retraining=False,
    )
    elapsed = time.time() - t0
    agg_odds = compute_all_metrics(preds_odds["y_true"], preds_odds["y_pred"], preds_odds["y_prob"])
    baseline_results["Odds Implied"] = agg_odds
    baseline_predictions["Odds Implied"] = preds_odds
    print(f"   Accuracy: {agg_odds['accuracy']:.4f} | Log-Loss: {agg_odds['log_loss']:.4f} | "
          f"Brier: {agg_odds['brier_score']:.4f} | AUC: {agg_odds['auc_roc']:.4f} | Time: {elapsed:.1f}s")
else:
    print("\n3. OddsImpliedModel -- SKIPPED (odds columns not found)")

# --- 4. Elo Model ---
if "home_team" in df.columns and "away_team" in df.columns:
    print("\n4. EloModel")
    t0 = time.time()
    elo_feature_cols = feature_cols + [year_col]
    for extra in ["home_team", "away_team", "season"]:
        if extra in df.columns and extra not in elo_feature_cols:
            elo_feature_cols.append(extra)

    results_elo, preds_elo = backtester.run(
        model_factory=EloModel,
        features_df=df[elo_feature_cols],
        target=target,
        needs_retraining=True,  # Elo needs to fit (build ratings) on training data
    )
    elapsed = time.time() - t0
    agg_elo = compute_all_metrics(preds_elo["y_true"], preds_elo["y_pred"], preds_elo["y_prob"])
    baseline_results["Elo"] = agg_elo
    baseline_predictions["Elo"] = preds_elo
    print(f"   Accuracy: {agg_elo['accuracy']:.4f} | Log-Loss: {agg_elo['log_loss']:.4f} | "
          f"Brier: {agg_elo['brier_score']:.4f} | AUC: {agg_elo['auc_roc']:.4f} | Time: {elapsed:.1f}s")

print("\nBaseline evaluation complete.")
display(compare_models(baseline_results))

In [ ]:
# ============================================================
# Cell 4: Train and evaluate classical models
# ============================================================

# Prepare a clean numeric feature matrix (fill NaNs for tree models)
df_model = df[feature_cols + [year_col]].copy()

# Fill NaN with column median for classical models
for col in feature_cols:
    if df_model[col].isnull().any():
        df_model[col] = df_model[col].fillna(df_model[col].median())

classical_results = {}
classical_predictions = {}
trained_models = {}  # Store last-fold trained model for SHAP analysis

model_names = [
    ("LogReg", "logistic_regression"),
    ("Random Forest", "random_forest"),
    ("XGBoost", "xgboost"),
    ("LightGBM", "lightgbm"),
]

print("Training classical models with walk-forward backtesting...")
print("=" * 60)

for display_name, model_key in model_names:
    print(f"\nTraining {display_name}...")
    t0 = time.time()

    try:
        # Model factory: returns a fresh unfitted model each fold
        factory = lambda mk=model_key: get_model(mk)

        results_df, preds_df = backtester.run(
            model_factory=factory,
            features_df=df_model,
            target=target,
            needs_retraining=True,
        )

        elapsed = time.time() - t0

        agg = compute_all_metrics(preds_df["y_true"], preds_df["y_pred"], preds_df["y_prob"])
        classical_results[display_name] = agg
        classical_predictions[display_name] = preds_df

        # Train a final model on all data up to the last test year for SHAP
        final_model = get_model(model_key)
        last_test_year = max(test_years)
        train_mask = df_model[year_col] < last_test_year
        X_final = df_model.loc[train_mask, feature_cols]
        y_final = target.loc[train_mask]
        final_model.fit(X_final, y_final)
        trained_models[display_name] = final_model

        print(f"   Accuracy: {agg['accuracy']:.4f} | Log-Loss: {agg['log_loss']:.4f} | "
              f"Brier: {agg['brier_score']:.4f} | AUC: {agg['auc_roc']:.4f} | Time: {elapsed:.1f}s")

        # Per-year breakdown
        print(f"   Per-year accuracy: ", end="")
        for yr in sorted(results_df.index):
            acc = results_df.loc[yr, "accuracy"] if "accuracy" in results_df.columns else None
            if acc is not None:
                print(f"{yr}={acc:.3f}", end="  ")
        print()

    except Exception as e:
        print(f"   ERROR: {e}")
        continue

print("\nClassical model evaluation complete.")

In [ ]:
# ============================================================
# Cell 5: Results comparison table
# ============================================================

# Combine all results
all_results = {**baseline_results, **classical_results}
all_predictions = {**baseline_predictions, **classical_predictions}

comparison_df = compare_models(all_results)

print("MODEL COMPARISON (Walk-Forward Backtest)")
print("=" * 70)
display(
    comparison_df.style
    .format("{:.4f}")
    .background_gradient(subset=["accuracy"], cmap="RdYlGn")
    .background_gradient(subset=["log_loss"], cmap="RdYlGn_r")
    .background_gradient(subset=["brier_score"], cmap="RdYlGn_r")
    .background_gradient(subset=["auc_roc"], cmap="RdYlGn")
)

# Visual comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

metrics_to_plot = [
    ("accuracy", "Accuracy (Higher is Better)", True),
    ("log_loss", "Log-Loss (Lower is Better)", False),
    ("brier_score", "Brier Score (Lower is Better)", False),
    ("auc_roc", "AUC-ROC (Higher is Better)", True),
]

for ax, (metric, title, higher_better) in zip(axes.ravel(), metrics_to_plot):
    if metric not in comparison_df.columns:
        continue
    vals = comparison_df[metric].sort_values(ascending=not higher_better)

    # Color: best model in green, worst in red
    colors = []
    for v in vals.values:
        if v == vals.values[0]:  # Best
            colors.append("#2ca02c")
        elif v == vals.values[-1]:  # Worst
            colors.append("#d62728")
        else:
            colors.append("#1f77b4")

    ax.barh(vals.index, vals.values, color=colors, edgecolor="white")
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.set_xlabel(metric.replace("_", " ").title())

    # Annotate values
    for i, v in enumerate(vals.values):
        ax.text(v, i, f"  {v:.4f}", va="center", fontsize=8)

fig.suptitle("Model Performance Comparison", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

# Identify best model
best_model_name = comparison_df["accuracy"].idxmax()
best_metrics = comparison_df.loc[best_model_name]
print(f"\nBest model by accuracy: {best_model_name}")
print(f"  Accuracy:    {best_metrics['accuracy']:.4f}")
print(f"  Log-Loss:    {best_metrics['log_loss']:.4f}")
print(f"  Brier Score: {best_metrics['brier_score']:.4f}")
print(f"  AUC-ROC:     {best_metrics['auc_roc']:.4f}")

In [ ]:
# ============================================================
# Cell 6: Calibration plots for top models
# ============================================================
from sklearn.calibration import calibration_curve

# Select top 4 models by accuracy for calibration analysis
top_models = comparison_df["accuracy"].sort_values(ascending=False).head(4).index.tolist()

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.ravel()

for i, model_name in enumerate(top_models):
    ax = axes[i]
    preds_df = all_predictions.get(model_name)
    if preds_df is None or preds_df.empty:
        ax.text(0.5, 0.5, f"{model_name}\nNo predictions", transform=ax.transAxes, ha="center")
        continue

    y_true = preds_df["y_true"].values
    y_prob = preds_df["y_prob"].values

    # Calibration curve
    try:
        fraction_pos, mean_predicted = calibration_curve(
            y_true, y_prob, n_bins=10, strategy="uniform"
        )
    except ValueError:
        fraction_pos, mean_predicted = np.array([]), np.array([])

    # Plot reliability diagram
    ax.plot([0, 1], [0, 1], "k--", linewidth=0.8, label="Perfectly calibrated")
    if len(fraction_pos) > 0:
        ax.plot(mean_predicted, fraction_pos, "s-", color="#1f77b4",
                label=f"{model_name}")

    # Compute ECE
    from evaluation.metrics import compute_ece
    ece = compute_ece(y_true, y_prob)

    ax.set_xlabel("Mean predicted probability")
    ax.set_ylabel("Fraction of positives")
    ax.set_title(f"{model_name} (ECE={ece:.4f})", fontsize=11, fontweight="bold")
    ax.legend(loc="lower right", fontsize=9)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3)

    # Inset histogram of predicted probabilities
    ax_inset = ax.inset_axes([0.6, 0.05, 0.35, 0.25])
    ax_inset.hist(y_prob, bins=20, color="#1f77b4", alpha=0.6, edgecolor="white")
    ax_inset.set_xlabel("P(home)", fontsize=7)
    ax_inset.set_ylabel("Count", fontsize=7)
    ax_inset.tick_params(labelsize=6)

fig.suptitle("Calibration Plots for Top Models", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Cell 7: Feature importance (SHAP) for best tree model
# ============================================================

# Find the best tree-based model for SHAP analysis
tree_model_names = ["XGBoost", "LightGBM", "Random Forest"]
best_tree_name = None
best_tree_acc = 0

for name in tree_model_names:
    if name in classical_results:
        acc = classical_results[name]["accuracy"]
        if acc > best_tree_acc:
            best_tree_acc = acc
            best_tree_name = name

if best_tree_name and best_tree_name in trained_models:
    print(f"Computing SHAP values for {best_tree_name} (accuracy={best_tree_acc:.4f})...")

    try:
        from modelling.interpretability import compute_shap_values, plot_shap_summary

        # Use the last test fold data for SHAP analysis
        last_test_year = max(test_years)
        test_mask = df_model[year_col] == last_test_year
        X_shap = df_model.loc[test_mask, feature_cols]

        # Compute SHAP values
        shap_values = compute_shap_values(
            trained_models[best_tree_name],
            X_shap,
            feature_names=feature_cols,
        )

        # SHAP summary plot
        import shap
        fig = plt.figure(figsize=(12, 10))
        shap.summary_plot(
            shap_values,
            X_shap,
            feature_names=feature_cols,
            max_display=20,
            show=False,
        )
        plt.title(f"SHAP Summary Plot - {best_tree_name}", fontsize=14, fontweight="bold")
        plt.tight_layout()
        plt.show()

        # Mean absolute SHAP values (bar chart)
        mean_abs_shap = np.abs(shap_values).mean(axis=0)
        shap_importance = pd.Series(mean_abs_shap, index=feature_cols).sort_values(ascending=False)

        fig, ax = plt.subplots(figsize=(10, 8))
        top20 = shap_importance.head(20)
        ax.barh(range(len(top20)), top20.values[::-1],
                color=sns.color_palette("viridis", 20), edgecolor="white")
        ax.set_yticks(range(len(top20)))
        ax.set_yticklabels(top20.index[::-1], fontsize=9)
        ax.set_xlabel("Mean |SHAP value|")
        ax.set_title(f"Top 20 Feature Importances (SHAP) - {best_tree_name}",
                     fontsize=14, fontweight="bold")
        plt.tight_layout()
        plt.show()

    except ImportError:
        print("shap package not installed. Falling back to built-in feature importance.")

        # Use built-in feature importance from the tree model
        model = trained_models[best_tree_name]
        if hasattr(model, "feature_importances_"):
            importances = model.feature_importances_
        elif hasattr(model, "named_steps"):
            importances = model.named_steps["clf"].feature_importances_
        else:
            importances = None

        if importances is not None:
            imp_df = pd.Series(importances, index=feature_cols).sort_values(ascending=False)
            top20 = imp_df.head(20)

            fig, ax = plt.subplots(figsize=(10, 8))
            ax.barh(range(len(top20)), top20.values[::-1],
                    color="#2ca02c", edgecolor="white")
            ax.set_yticks(range(len(top20)))
            ax.set_yticklabels(top20.index[::-1], fontsize=9)
            ax.set_xlabel("Feature Importance")
            ax.set_title(f"Top 20 Feature Importances - {best_tree_name}",
                         fontsize=14, fontweight="bold")
            plt.tight_layout()
            plt.show()
else:
    print("No tree-based model available for feature importance analysis.")

In [ ]:
# ============================================================
# Cell 8: Summary and best model selection
# ============================================================

print("=" * 70)
print("FINAL MODEL COMPARISON SUMMARY")
print("=" * 70)

# Display final comparison
final_comparison = compare_models(all_results)
display(final_comparison.style.format("{:.4f}"))

# Determine best model overall (using a composite score)
# Weight: accuracy (40%), log_loss (30%), brier (15%), AUC (15%)
if not final_comparison.empty:
    # Normalize each metric to [0, 1] and compute weighted score
    score_df = final_comparison.copy()

    for col in ["accuracy", "auc_roc"]:
        if col in score_df.columns:
            rng = score_df[col].max() - score_df[col].min()
            score_df[f"{col}_norm"] = (score_df[col] - score_df[col].min()) / rng if rng > 0 else 0.5

    for col in ["log_loss", "brier_score"]:
        if col in score_df.columns:
            rng = score_df[col].max() - score_df[col].min()
            score_df[f"{col}_norm"] = (score_df[col].max() - score_df[col]) / rng if rng > 0 else 0.5

    weights = {"accuracy_norm": 0.40, "log_loss_norm": 0.30,
               "brier_score_norm": 0.15, "auc_roc_norm": 0.15}

    score_df["composite_score"] = sum(
        score_df.get(col, 0) * w
        for col, w in weights.items()
        if col in score_df.columns
    )

    best_overall = score_df["composite_score"].idxmax()

    print(f"\nBest model (composite weighted score): {best_overall}")
    print(f"  Composite score: {score_df.loc[best_overall, 'composite_score']:.4f}")
    print(f"  Accuracy:        {final_comparison.loc[best_overall, 'accuracy']:.4f}")
    print(f"  Log-Loss:        {final_comparison.loc[best_overall, 'log_loss']:.4f}")
    print(f"  Brier Score:     {final_comparison.loc[best_overall, 'brier_score']:.4f}")
    print(f"  AUC-ROC:         {final_comparison.loc[best_overall, 'auc_roc']:.4f}")

    # Save the best model predictions for error analysis
    best_preds = all_predictions.get(best_overall)
    if best_preds is not None:
        preds_save_path = OUTPUTS_DIR / "best_model_predictions.parquet"
        best_preds.to_parquet(preds_save_path, index=False)
        print(f"\nBest model predictions saved to: {preds_save_path}")

    # Save the trained model if it is a classical model
    if best_overall in trained_models:
        from modelling.model_registry import save_model
        model_path = save_model(
            trained_models[best_overall],
            name=f"best_{best_overall.lower().replace(' ', '_')}",
            metadata={
                "feature_cols": feature_cols,
                "metrics": all_results[best_overall],
                "test_years": test_years,
            },
        )
        print(f"Best model saved to registry: {model_path}")

    print("\nKey takeaways:")
    print(f"  - Best baseline: {max(baseline_results, key=lambda k: baseline_results[k]['accuracy'])} "
          f"({max(r['accuracy'] for r in baseline_results.values()):.4f})")
    if classical_results:
        print(f"  - Best classical model: {max(classical_results, key=lambda k: classical_results[k]['accuracy'])} "
              f"({max(r['accuracy'] for r in classical_results.values()):.4f})")
        baseline_best = max(r["accuracy"] for r in baseline_results.values())
        classical_best = max(r["accuracy"] for r in classical_results.values())
        improvement = classical_best - baseline_best
        print(f"  - Improvement over best baseline: {improvement:+.4f} ({improvement * 100:+.1f} pp)")